# Exploring the Impact of Disruptions on Supply Chains
**PURE – Program for Undergraduate Research, Sabancı University (Fall 2025–26)**
Görkem Kocamanoğlu · Mustafa Mert Mutlu · Simay Karaca — Supervisor: Assoc. Prof. Fatma Tevhide Altekin

This notebook reproduces the analysis pipeline on the anonymised sample network of the
IFAC MIM 2025 Data Challenge (Complex Network Analysis track). The raw data is **not** included in this
repository; see `data/README.md` for how to obtain it.

> **Revision note (v2):** the original Colab notebook read about half of the node file incorrectly
> (rows wrapped in extra quotes were parsed as a single field), so only ~22% of the edges entered the graph.
> This version fixes the parsing and re-assigns source nodes after cycle-breaking. See the README for how the
> results differ from the original report.

In [1]:
import sys; sys.path.append('../src')
import pandas as pd, networkx as nx
from pipeline import load_nodes, build_dag, weighted_risk_index, starvation_simulation
import figures

NODES, EDGES = '../data/nodes.csv', '../data/edges.csv'

## 1. Load data and build a multi-echelon DAG

In [2]:
nodes = load_nodes(NODES)
edges = pd.read_csv(EDGES)
G, stats = build_dag(nodes, edges)
stats

{'edges_in_file': 6529,
 'edges_loaded': 6527,
 'removed_backward': 1458,
 'removed_cycle': 583,
 'final_edges': 4486,
 'nodes': 1035,
 'stage_counts': {'Retail': 568, 'Manuf': 220, 'Part': 135, 'Dist': 112}}

Nodes are classified into four Willems-style echelons (Part → Manuf → Dist → Retail) from product keywords,
refined by topology (no supplier ⇒ Part, no customer ⇒ Retail). Edges that flow backwards in this hierarchy are removed,
remaining cycles are broken with DFS, and nodes left without any supplier are re-labelled as entry points (Part).

## 2. Weighted Risk Index

In [3]:
wri = weighted_risk_index(G, w=(0.5, 0.3, 0.2))
wri.head(10)[['id', 'stage', 'downstream', 'betweenness', 'out_degree', 'risk_index']]

,id,stage,downstream,betweenness,out_degree,risk_index
0,n793,Manuf,684,0.017688,155,0.843029
1,n161,Part,823,0.007750,128,0.709336
2,n221,Part,934,0.003185,105,0.657901
3,n613,Part,871,0.002269,132,0.645616
4,n708,Part,881,0.005372,71,0.624543
5,n1033,Manuf,725,0.011941,43,0.621596
6,n523,Part,980,0.000839,68,0.593445
7,n163,Part,996,0.001999,37,0.581145
8,n321,Manuf,540,0.010801,94,0.575289
9,n801,Manuf,326,0.016573,97,0.569739


In [4]:
figures.risk_index_by_stage(wri, '../figures/risk_index_top10.png')

## 3. Supply-starvation simulation

Each node is removed in turn; a node is *starved* if it can no longer reach any Part node.
**Criticality** = number of nodes a failure starves. **Vulnerability** = number of scenarios in which a node is starved.

In [5]:
sim = starvation_simulation(G)
sim.sort_values('criticality', ascending=False).head(10)

,id,criticality,vulnerability,stage
793,n793,85,0,Manuf
801,n801,78,0,Manuf
161,n161,40,0,Part
321,n321,30,0,Manuf
38,n38,28,0,Manuf
221,n221,26,0,Part
523,n523,18,0,Part
173,n173,17,0,Manuf
953,n953,13,0,Part
163,n163,13,0,Part


In [6]:
sim.sort_values('vulnerability', ascending=False).head(10)

,id,criticality,vulnerability,stage
433,n433,0,2,Retail
779,n779,0,2,Dist
728,n728,0,2,Retail
284,n284,0,2,Manuf
810,n810,0,1,Retail
807,n807,0,1,Manuf
387,n387,0,1,Retail
388,n388,0,1,Retail
390,n390,0,1,Retail
393,n393,0,1,Retail


In [7]:
print('Share of nodes never starved by any single failure:', round((sim.vulnerability == 0).mean(), 3))
print('Max share of the network starved by one failure:', round(sim.criticality.max() / G.number_of_nodes(), 3))
print('corr(criticality, vulnerability):', round(sim.criticality.corr(sim.vulnerability), 3))

Share of nodes never starved by any single failure: 0.586
Max share of the network starved by one failure: 0.082
corr(criticality, vulnerability): -0.081


In [8]:
figures.top_criticality_bar(sim, '../figures/top_criticality.png')
top_node = sim.sort_values('criticality', ascending=False).iloc[0]['id']
figures.critical_node_neighbourhood(G, top_node, '../figures/critical_node_neighbourhood.png')

## 4. Takeaways
- Risk is concentrated: a handful of manufacturing (vehicle-part) and raw-material nodes dominate both the
  Weighted Risk Index and the starvation simulation.
- The network is redundant at the level of single failures: most firms keep at least one path to raw materials
  when any one node fails.
- Criticality and vulnerability are close to uncorrelated: the firms that would hurt the network most are not the
  ones most exposed to upstream failures.